### Imports

In [1]:
import json
import os
import pandas as pd

from utils_MS import *

# %load_ext autotime

/home/ealvarez/miniconda3/envs/graph_matching/lib/python3.10/site-packages/outdated/utils.py:14: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.5.3, the latest is 0.6.0.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(


In [2]:
# def run(params):

### Parameters

In [3]:
""" try:
    dir = os.path.dirname(os.path.abspath(__file__))
except:
    dir = os.getcwd()
print(dir) """

' try:\n    dir = os.path.dirname(os.path.abspath(__file__))\nexcept:\n    dir = os.getcwd()\nprint(dir) '

In [4]:
dict_dataset = {
    1: ["Mentos_2_process_NormalizationFiltered_format", ["Orange"]], # new mentos
    2: ["deybis_filter_september_2br_3ar_format", ["SecoAmazonas"]],
    3: ["deybis_filter_december_2br_3ar_format", ["SecoAmazonas"]],
    4: ["deybis_filter_september_2br_10ar_format", ["SecoAmazonas"]],
    5: ["deybis_filter_december_2br_10ar_format", ["SecoAmazonas"]],
    6: ["deybis_filter_september_min_2br_3ar_format", ["SecoAmazonas"]],
    7: ["deybis_filter_december_min_2br_3ar_format", ["SecoAmazonas"]],
}
dataset = dict_dataset[7] # change
dataset

['deybis_filter_december_min_2br_3ar_format', ['SecoAmazonas']]

In [5]:
params = {
    "exp": "exp6", # Change
    "methods": ["t-gae"], # ["vgae-base", "argva-base", "vgae-line", "dgi-tran", "t-gae"],
    "data_variations": ["none"],
    "has_transformation": False, # True or False
    "controls": dataset[1],
    "dimension": 32,
    "threshold_corr": 0.5,
    "threshold_log2": 0,
    "alpha": 0.05,
    "iterations": 1,
    "raw_data_file": dataset[0],
    "groups_id_no": ["Blank", "QC", "Std"],
    "sensitivity": False, # False: f1 (selectivity), True: f1 (selectivity), f2 (sensitivity)
    "obs": "",
    "seeds": [41, 42, 43, 44, 45, 46],
    
    "from": "python",
    "cuda": 1,
    "epochs": 100,
    "lr": 0.0001,
    "weight_decay": 1e-4,
    "patience": 10,
    "contamination": 0.1, # float in (0., 0.5)
    "n_jobs": 1, # -1 all
}

In [6]:
""" dir_path = "experiments/output"
res = sorted(os.listdir(dir_path))
n = len(res)
exp = "exp{}".format(n) """

exp = str(params["exp"])
exp

'exp6'

### Load dataset

In [7]:
# load dataset groups
if params["from"] == "python":
    df_raw = pd.read_csv("experiments/raw_data/{}.csv".format(params["raw_data_file"]), delimiter="|")
elif params["from"] == "drf":
    df_raw = pd.read_csv("{}".format(params["raw_data_file"]), delimiter="|") # from DRF
df_raw

,Alignment ID,Average Rt,Average Mz,Metabolite name,SecoAmazonas_1.1,SecoAmazonas_1.2,SecoAmazonas_1.3,SecoAmazonas_2.1,SecoAmazonas_2.2,SecoAmazonas_2.3,...,FrescoCusco_1.3,FrescoCusco_2.1,FrescoCusco_2.2,FrescoCusco_2.3,FrescoSanMartin_1.1,FrescoSanMartin_1.2,FrescoSanMartin_1.3,FrescoSanMartin_2.1,FrescoSanMartin_2.2,FrescoSanMartin_2.3
0,0,2.120,152.05702,unknown,1.947976e+06,2.145146e+06,1.889418e+06,1.756070e+06,1.281853e+06,1.737510e+06,...,2.788924e+05,2.495039e+05,4.248775e+05,3.098800e+05,2.888458e+05,4.263454e+05,2.601564e+05,2.230069e+05,2.389870e+05,1.698564e+05
1,1,2.125,257.96816,unknown,1.241109e+07,2.402011e+06,2.862010e+06,2.414489e+06,3.688394e+06,4.478066e+06,...,1.291696e+06,4.211488e+06,4.924348e+06,1.276892e+07,1.048620e+07,4.959735e+06,1.233451e+07,5.116804e+06,8.661161e+05,1.246542e+07
2,2,2.128,207.98572,unknown,5.125119e+06,6.072819e+06,5.915750e+06,7.670063e+06,1.112191e+07,6.413633e+06,...,6.876271e+06,5.189219e+06,5.727825e+06,3.114297e+06,4.682256e+06,1.237203e+07,5.174974e+06,6.716172e+06,7.540170e+06,7.938045e+06
3,3,2.133,217.96001,unknown,1.784870e+06,1.576395e+06,9.802001e+05,1.714031e+06,1.990814e+06,2.014875e+06,...,1.598919e+06,1.272286e+06,1.024954e+06,1.517614e+06,1.434988e+06,1.253071e+06,1.796903e+06,1.959247e+06,1.377944e+06,1.642902e+06
4,4,2.255,152.05702,unknown,1.947976e+06,2.145146e+06,1.889418e+06,1.756070e+06,1.281853e+06,1.737510e+06,...,2.788924e+05,2.495039e+05,4.248775e+05,3.098800e+05,2.888458e+05,4.263454e+05,2.601564e+05,2.230069e+05,2.389870e+05,1.698564e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,77,13.576,359.19705,unknown,2.946200e+07,3.335107e+07,3.520060e+07,2.942578e+07,3.004647e+07,3.151865e+07,...,3.547148e+07,3.244064e+07,3.301073e+07,3.187038e+07,3.838530e+07,2.989014e+07,3.527329e+07,2.920353e+07,3.044284e+07,3.695492e+07
78,78,13.782,515.23988,unknown,3.533964e+06,4.744814e+06,3.217944e+06,3.319008e+06,4.928576e+06,3.432516e+06,...,2.779232e+06,3.833238e+06,4.436233e+06,4.582209e+06,4.422106e+06,3.285745e+06,4.329191e+06,3.609709e+06,4.725286e+06,3.467679e+06
79,79,14.886,667.23171,unknown,2.096873e+06,2.334722e+06,2.275303e+06,1.700928e+06,2.053808e+06,2.442797e+06,...,2.267375e+06,2.866012e+06,2.191374e+06,1.940903e+06,2.732096e+06,1.846165e+06,2.074487e+06,2.443682e+06,1.844960e+06,3.324116e+06
80,80,15.129,599.31616,unknown,1.193444e+06,4.534499e+06,5.824158e+06,4.594740e+06,8.490425e+05,3.976399e+06,...,4.308660e+06,1.364190e+06,1.602308e+06,3.487020e+06,3.755398e+06,4.040052e+06,1.644414e+06,3.098998e+06,2.836007e+06,1.862349e+06


### Format dataset

In [8]:
# has transformation
columns_data = list(df_raw.columns)[4:]
if params["has_transformation"]:
    print("transformation")
    for column in columns_data:
        df_raw[column] = df_raw[column].apply(lambda x: 10**x)
df_raw

,Alignment ID,Average Rt,Average Mz,Metabolite name,SecoAmazonas_1.1,SecoAmazonas_1.2,SecoAmazonas_1.3,SecoAmazonas_2.1,SecoAmazonas_2.2,SecoAmazonas_2.3,...,FrescoCusco_1.3,FrescoCusco_2.1,FrescoCusco_2.2,FrescoCusco_2.3,FrescoSanMartin_1.1,FrescoSanMartin_1.2,FrescoSanMartin_1.3,FrescoSanMartin_2.1,FrescoSanMartin_2.2,FrescoSanMartin_2.3
0,0,2.120,152.05702,unknown,1.947976e+06,2.145146e+06,1.889418e+06,1.756070e+06,1.281853e+06,1.737510e+06,...,2.788924e+05,2.495039e+05,4.248775e+05,3.098800e+05,2.888458e+05,4.263454e+05,2.601564e+05,2.230069e+05,2.389870e+05,1.698564e+05
1,1,2.125,257.96816,unknown,1.241109e+07,2.402011e+06,2.862010e+06,2.414489e+06,3.688394e+06,4.478066e+06,...,1.291696e+06,4.211488e+06,4.924348e+06,1.276892e+07,1.048620e+07,4.959735e+06,1.233451e+07,5.116804e+06,8.661161e+05,1.246542e+07
2,2,2.128,207.98572,unknown,5.125119e+06,6.072819e+06,5.915750e+06,7.670063e+06,1.112191e+07,6.413633e+06,...,6.876271e+06,5.189219e+06,5.727825e+06,3.114297e+06,4.682256e+06,1.237203e+07,5.174974e+06,6.716172e+06,7.540170e+06,7.938045e+06
3,3,2.133,217.96001,unknown,1.784870e+06,1.576395e+06,9.802001e+05,1.714031e+06,1.990814e+06,2.014875e+06,...,1.598919e+06,1.272286e+06,1.024954e+06,1.517614e+06,1.434988e+06,1.253071e+06,1.796903e+06,1.959247e+06,1.377944e+06,1.642902e+06
4,4,2.255,152.05702,unknown,1.947976e+06,2.145146e+06,1.889418e+06,1.756070e+06,1.281853e+06,1.737510e+06,...,2.788924e+05,2.495039e+05,4.248775e+05,3.098800e+05,2.888458e+05,4.263454e+05,2.601564e+05,2.230069e+05,2.389870e+05,1.698564e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,77,13.576,359.19705,unknown,2.946200e+07,3.335107e+07,3.520060e+07,2.942578e+07,3.004647e+07,3.151865e+07,...,3.547148e+07,3.244064e+07,3.301073e+07,3.187038e+07,3.838530e+07,2.989014e+07,3.527329e+07,2.920353e+07,3.044284e+07,3.695492e+07
78,78,13.782,515.23988,unknown,3.533964e+06,4.744814e+06,3.217944e+06,3.319008e+06,4.928576e+06,3.432516e+06,...,2.779232e+06,3.833238e+06,4.436233e+06,4.582209e+06,4.422106e+06,3.285745e+06,4.329191e+06,3.609709e+06,4.725286e+06,3.467679e+06
79,79,14.886,667.23171,unknown,2.096873e+06,2.334722e+06,2.275303e+06,1.700928e+06,2.053808e+06,2.442797e+06,...,2.267375e+06,2.866012e+06,2.191374e+06,1.940903e+06,2.732096e+06,1.846165e+06,2.074487e+06,2.443682e+06,1.844960e+06,3.324116e+06
80,80,15.129,599.31616,unknown,1.193444e+06,4.534499e+06,5.824158e+06,4.594740e+06,8.490425e+05,3.976399e+06,...,4.308660e+06,1.364190e+06,1.602308e+06,3.487020e+06,3.755398e+06,4.040052e+06,1.644414e+06,3.098998e+06,2.836007e+06,1.862349e+06


In [9]:
# concat
df_join_raw = pd.concat([
    df_raw.iloc[:, :]], axis=1)
df_join_raw.set_index("Alignment ID", inplace=True)
df_join_raw

,Average Rt,Average Mz,Metabolite name,SecoAmazonas_1.1,SecoAmazonas_1.2,SecoAmazonas_1.3,SecoAmazonas_2.1,SecoAmazonas_2.2,SecoAmazonas_2.3,SecoCusco_1.1,...,FrescoCusco_1.3,FrescoCusco_2.1,FrescoCusco_2.2,FrescoCusco_2.3,FrescoSanMartin_1.1,FrescoSanMartin_1.2,FrescoSanMartin_1.3,FrescoSanMartin_2.1,FrescoSanMartin_2.2,FrescoSanMartin_2.3
Alignment ID,,,,,,,,,,,,,,,,,,,,,
0,2.120,152.05702,unknown,1.947976e+06,2.145146e+06,1.889418e+06,1.756070e+06,1.281853e+06,1.737510e+06,2.758043e+06,...,2.788924e+05,2.495039e+05,4.248775e+05,3.098800e+05,2.888458e+05,4.263454e+05,2.601564e+05,2.230069e+05,2.389870e+05,1.698564e+05
1,2.125,257.96816,unknown,1.241109e+07,2.402011e+06,2.862010e+06,2.414489e+06,3.688394e+06,4.478066e+06,4.357772e+06,...,1.291696e+06,4.211488e+06,4.924348e+06,1.276892e+07,1.048620e+07,4.959735e+06,1.233451e+07,5.116804e+06,8.661161e+05,1.246542e+07
2,2.128,207.98572,unknown,5.125119e+06,6.072819e+06,5.915750e+06,7.670063e+06,1.112191e+07,6.413633e+06,5.885871e+06,...,6.876271e+06,5.189219e+06,5.727825e+06,3.114297e+06,4.682256e+06,1.237203e+07,5.174974e+06,6.716172e+06,7.540170e+06,7.938045e+06
3,2.133,217.96001,unknown,1.784870e+06,1.576395e+06,9.802001e+05,1.714031e+06,1.990814e+06,2.014875e+06,9.986963e+05,...,1.598919e+06,1.272286e+06,1.024954e+06,1.517614e+06,1.434988e+06,1.253071e+06,1.796903e+06,1.959247e+06,1.377944e+06,1.642902e+06
4,2.255,152.05702,unknown,1.947976e+06,2.145146e+06,1.889418e+06,1.756070e+06,1.281853e+06,1.737510e+06,2.758043e+06,...,2.788924e+05,2.495039e+05,4.248775e+05,3.098800e+05,2.888458e+05,4.263454e+05,2.601564e+05,2.230069e+05,2.389870e+05,1.698564e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,13.576,359.19705,unknown,2.946200e+07,3.335107e+07,3.520060e+07,2.942578e+07,3.004647e+07,3.151865e+07,3.869054e+07,...,3.547148e+07,3.244064e+07,3.301073e+07,3.187038e+07,3.838530e+07,2.989014e+07,3.527329e+07,2.920353e+07,3.044284e+07,3.695492e+07
78,13.782,515.23988,unknown,3.533964e+06,4.744814e+06,3.217944e+06,3.319008e+06,4.928576e+06,3.432516e+06,5.062720e+06,...,2.779232e+06,3.833238e+06,4.436233e+06,4.582209e+06,4.422106e+06,3.285745e+06,4.329191e+06,3.609709e+06,4.725286e+06,3.467679e+06
79,14.886,667.23171,unknown,2.096873e+06,2.334722e+06,2.275303e+06,1.700928e+06,2.053808e+06,2.442797e+06,2.437916e+06,...,2.267375e+06,2.866012e+06,2.191374e+06,1.940903e+06,2.732096e+06,1.846165e+06,2.074487e+06,2.443682e+06,1.844960e+06,3.324116e+06


In [10]:
# split
df_join_raw = df_join_raw.rename_axis(None)
# df_join_raw = df_join_raw.iloc[:, 2:]
df_join_raw

,Average Rt,Average Mz,Metabolite name,SecoAmazonas_1.1,SecoAmazonas_1.2,SecoAmazonas_1.3,SecoAmazonas_2.1,SecoAmazonas_2.2,SecoAmazonas_2.3,SecoCusco_1.1,...,FrescoCusco_1.3,FrescoCusco_2.1,FrescoCusco_2.2,FrescoCusco_2.3,FrescoSanMartin_1.1,FrescoSanMartin_1.2,FrescoSanMartin_1.3,FrescoSanMartin_2.1,FrescoSanMartin_2.2,FrescoSanMartin_2.3
0,2.120,152.05702,unknown,1.947976e+06,2.145146e+06,1.889418e+06,1.756070e+06,1.281853e+06,1.737510e+06,2.758043e+06,...,2.788924e+05,2.495039e+05,4.248775e+05,3.098800e+05,2.888458e+05,4.263454e+05,2.601564e+05,2.230069e+05,2.389870e+05,1.698564e+05
1,2.125,257.96816,unknown,1.241109e+07,2.402011e+06,2.862010e+06,2.414489e+06,3.688394e+06,4.478066e+06,4.357772e+06,...,1.291696e+06,4.211488e+06,4.924348e+06,1.276892e+07,1.048620e+07,4.959735e+06,1.233451e+07,5.116804e+06,8.661161e+05,1.246542e+07
2,2.128,207.98572,unknown,5.125119e+06,6.072819e+06,5.915750e+06,7.670063e+06,1.112191e+07,6.413633e+06,5.885871e+06,...,6.876271e+06,5.189219e+06,5.727825e+06,3.114297e+06,4.682256e+06,1.237203e+07,5.174974e+06,6.716172e+06,7.540170e+06,7.938045e+06
3,2.133,217.96001,unknown,1.784870e+06,1.576395e+06,9.802001e+05,1.714031e+06,1.990814e+06,2.014875e+06,9.986963e+05,...,1.598919e+06,1.272286e+06,1.024954e+06,1.517614e+06,1.434988e+06,1.253071e+06,1.796903e+06,1.959247e+06,1.377944e+06,1.642902e+06
4,2.255,152.05702,unknown,1.947976e+06,2.145146e+06,1.889418e+06,1.756070e+06,1.281853e+06,1.737510e+06,2.758043e+06,...,2.788924e+05,2.495039e+05,4.248775e+05,3.098800e+05,2.888458e+05,4.263454e+05,2.601564e+05,2.230069e+05,2.389870e+05,1.698564e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,13.576,359.19705,unknown,2.946200e+07,3.335107e+07,3.520060e+07,2.942578e+07,3.004647e+07,3.151865e+07,3.869054e+07,...,3.547148e+07,3.244064e+07,3.301073e+07,3.187038e+07,3.838530e+07,2.989014e+07,3.527329e+07,2.920353e+07,3.044284e+07,3.695492e+07
78,13.782,515.23988,unknown,3.533964e+06,4.744814e+06,3.217944e+06,3.319008e+06,4.928576e+06,3.432516e+06,5.062720e+06,...,2.779232e+06,3.833238e+06,4.436233e+06,4.582209e+06,4.422106e+06,3.285745e+06,4.329191e+06,3.609709e+06,4.725286e+06,3.467679e+06
79,14.886,667.23171,unknown,2.096873e+06,2.334722e+06,2.275303e+06,1.700928e+06,2.053808e+06,2.442797e+06,2.437916e+06,...,2.267375e+06,2.866012e+06,2.191374e+06,1.940903e+06,2.732096e+06,1.846165e+06,2.074487e+06,2.443682e+06,1.844960e+06,3.324116e+06
80,15.129,599.31616,unknown,1.193444e+06,4.534499e+06,5.824158e+06,4.594740e+06,8.490425e+05,3.976399e+06,4.301930e+06,...,4.308660e+06,1.364190e+06,1.602308e+06,3.487020e+06,3.755398e+06,4.040052e+06,1.644414e+06,3.098998e+06,2.836007e+06,1.862349e+06


In [11]:
# get groups name
groups_id_no = params["groups_id_no"]
groups_id = []
for item in df_join_raw.iloc[:, 3:].columns.values:
    group_id = item.split("_")[0]
    if group_id not in groups_id and group_id not in groups_id_no:
        groups_id.append(group_id)
groups_id

['SecoAmazonas',
 'SecoCusco',
 'SecoSanMartin',
 'FrescoAmazonas',
 'FrescoCusco',
 'FrescoSanMartin']

In [12]:
# delete no sample columns
""" columns_delete = [columna for columna in df_join_raw.columns if columna.split("_")[0] in columns_no_sample]
df_join_raw.drop(columns_delete, axis=1, inplace=True)
df_join_raw """

' columns_delete = [columna for columna in df_join_raw.columns if columna.split("_")[0] in columns_no_sample]\ndf_join_raw.drop(columns_delete, axis=1, inplace=True)\ndf_join_raw '

In [13]:
# get subgroups names

""" def get_subgroups_id(df_join_raw, groups, by_group=False):
    dict_groups_id = {}
    for group in groups:
        # get group
        if by_group:
            dict_groups_id[group] = ["1"]
        else:
            columns = list(df_join_raw.filter(like=group).columns)
            subgroups = [item.split("{}_".format(group))[1].split(".")[0] for item in columns]
            subgroups = np.unique(subgroups)
            dict_groups_id[group] = subgroups.tolist()
    return dict_groups_id """

subgroups_id = get_subgroups_id(df_join_raw, groups_id)
subgroups_id

{'SecoAmazonas': ['1', '2'],
 'SecoCusco': ['1', '2'],
 'SecoSanMartin': ['1', '2'],
 'FrescoAmazonas': ['1', '2'],
 'FrescoCusco': ['1', '2'],
 'FrescoSanMartin': ['1', '2']}

In [14]:
# count analtical repetitions
# df_join_raw.filter(like="AA_1.")

In [15]:
# check distribution

In [16]:
""" x = df_join_raw.iloc[2, 3:]
print(x.min(), x.max(), x.mean())
x.hist(bins=200) """

' x = df_join_raw.iloc[2, 3:]\nprint(x.min(), x.max(), x.mean())\nx.hist(bins=200) '

In [17]:
# f_join_raw.iloc[:, 5].hist(bins=100)

In [18]:
params["controls"], groups_id

(['SecoAmazonas'],
 ['SecoAmazonas',
  'SecoCusco',
  'SecoSanMartin',
  'FrescoAmazonas',
  'FrescoCusco',
  'FrescoSanMartin'])

In [19]:
# get groups combination
groups = []
controls = params["controls"]

groups = []
for control in controls:
    for group_id in groups_id:
        if control != group_id:
            groups.append([control, group_id])
print(groups)

[['SecoAmazonas', 'SecoCusco'], ['SecoAmazonas', 'SecoSanMartin'], ['SecoAmazonas', 'FrescoAmazonas'], ['SecoAmazonas', 'FrescoCusco'], ['SecoAmazonas', 'FrescoSanMartin']]


### Create folders

In [20]:
# create experiments folder
try: 
    os.mkdir("experiments/output/{}".format(exp))
    os.mkdir("experiments/output/{}/correlations".format(exp))
    os.mkdir("experiments/output/{}/preprocessing".format(exp))
    os.mkdir("experiments/output/{}/preprocessing/edges".format(exp))
    os.mkdir("experiments/output/{}/preprocessing/graphs_data".format(exp))
    os.mkdir("experiments/output/{}/loss".format(exp))
    os.mkdir("experiments/output/{}/node_embeddings".format(exp))
    os.mkdir("experiments/output/{}/common_nodes".format(exp))
    os.mkdir("experiments/output/{}/filter_raw".format(exp))
    os.mkdir("experiments/output/{}/plots".format(exp))
except OSError as error: 
    print(error)

[Errno 17] File exists: 'experiments/output/exp6'


### Save dataset and parameters

In [21]:
# save dataset
df_join_raw.to_csv("experiments/input/{}_raw.csv".format(exp), index=True)

# save parameters
parameters = {
    "exp": exp,
    "methods": params["methods"],
    "data_variations": params["data_variations"],
    "has_transformation": params["has_transformation"],
    "controls": params["controls"],
    "dimension": params["dimension"],
    "threshold_corr": params["threshold_corr"],
    "threshold_log2": params["threshold_log2"],
    "alpha": params["alpha"],
    "iterations": params["iterations"],
    "raw_data_file": params["raw_data_file"],
    "groups_id": groups_id,
    "subgroups_id": subgroups_id,
    "groups": groups,
    "groups_id_no": params["groups_id_no"],
    "sensitivity": params["sensitivity"],
    
    "from": params["from"],
    "cuda": params["cuda"],
    "epochs": params["epochs"],
    "lr": params["lr"],
    "weight_decay": params["weight_decay"],
    "patience": params["patience"],
    "contamination": params["contamination"],
    "n_jobs": params["n_jobs"],

    "seeds": params["seeds"],
    "obs": params["obs"]
}

with open("experiments/output/{}/parameters.json".format(exp), "w") as outfile:
    json.dump(parameters, outfile, indent=4)

In [22]:
experiments = {
    "exp": exp
}

with open("exp.json".format(experiments), "w") as outfile:
    json.dump(experiments, outfile, indent=4)

In [23]:
# return exp